# 13 · The unified model, seeds 123 and 2026, final configuration, one L4 session

**L4. Run it only after Dhanush's go-ahead** (≈ 7.5 compute units, estimated).

**The final configuration (CP4, 18 Nov 2026):** the context-crop unified model, `data.context_crop_share = 0.3333`, the `eviground_ctx` experiment, with the same schedule as seed 42 (20 epochs at most, early stop with patience 4).
- **Seeds 123 and 2026 train back to back in this one session.** The shards are staged once.
- **A1** (the 64-image overfit) checks the pipeline, not the seed, so CP3's record is reused, as in notebook 11.
- **The detection model's extra seeds** run later, and only if at least 10 CU remain afterwards.

**If the session drops, run all cells again:**
- a seed whose `summary.json` exists is skipped;
- a seed in progress resumes from `last.pt`. The trainer stops if the seed or the context share differs from the checkpoint's.

**After training**, each seed gets the CP3 checks and a comparison with seed 42's run on validation crops (each at its best epoch). The last cell writes the session's compute record and releases the runtime.

**Results** (send to Claude):
- `Paper 1/runs/checkpoints/training_health_unified_ctx_seed123.json` and `…_seed2026.json`;
- `Paper 1/runs/checkpoints/compute_unified_seeds.json`;
- the run folders `Paper 1/runs/eviground_ctx_unified_seed{123,2026}/`.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings (unified only; final configuration)
import datetime, json, logging, os, time
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
import torch
from eviground.config import load_config
from eviground.data import pack
from eviground.env import get_paths, session_info

PROFILE = 'unified'
SEEDS = [123, 2026]        # Dhanush, 18 Nov 2026: both in one session, unified first
REF_SEED = 42              # the gated run of the final configuration (CP4)
CONTEXT_SHARE = 0.3333     # the final configuration (CP4 fix, 17 Nov 2026)
CU_PER_HOUR = 1.7          # read the live rate in Colab's resource panel and update
T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:   # fake world, fake shards, tiny CLIP, CPU
    import shutil, tempfile
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_seeds_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
paths.ensure()
CONFIG = 'debug' if DRY else 'base'
HF_CACHE = paths.cache / 'hf' if DRY else Path('/content/hf_cache')
OVERRIDES = [f'model.semantic.cache_dir={HF_CACHE}', f'compute.cu_per_hour={CU_PER_HOUR}',
             f'data.context_crop_share={CONTEXT_SHARE}']
cfg = load_config(CONFIG, OVERRIDES)
if not DRY:
    assert torch.cuda.is_available(), 'the seeds need a GPU runtime (L4)'
    assert cfg.model.semantic.revision and cfg.model.semantic.weights_sha256, \
        'pin the CLIP commit and weights hash in configs/base.yaml'
assert not cfg.model.trace.get('channels_last', False), 'channels_last stays off (15 Nov)'
assert abs(float(cfg.data.context_crop_share) - CONTEXT_SHARE) < 1e-9, cfg.data.context_crop_share
REVISION = cfg.model.semantic.revision if not DRY else 'tiny'
CP3_EXPERIMENT = cfg.get('experiment', 'eviground')        # CP3's A1 record
EXPERIMENT = f'{CP3_EXPERIMENT}_ctx'                         # the final configuration's runs
def run_for(seed):
    return paths.runs / f'{EXPERIMENT}_{PROFILE}_seed{seed}'
REF_RUN = run_for(REF_SEED)
A1_FILE = paths.runs / f'{CP3_EXPERIMENT}_unified_seed42_overfit' / 'overfit_a1.json'
CK = paths.runs / 'checkpoints'
NOTEBOOK = '13 unified seeds'
print('seeds', SEEDS, '| commit', info['git_commit'], '| gpu', info.get('gpu_name'), '| CLIP revision', REVISION,
      '| context share', CONTEXT_SHARE, '| run folders', [run_for(s).name for s in SEEDS])

In [ ]:
# Cell 2b: pre-flight. Prints the runs and their estimated CU; stops before staging if an input is missing
# or the estimate exceeds CU_LEFT (the balance shown in Colab's resource panel; None: not checked)
from eviground.utils import session
CU_LEFT = None
EST_CU = 7.5   # 2 x 3.42 CU of training (the ctx run, 1.7 CU/h) + one staging and set-up
NEEDED = {'CP3 A1 record': A1_FILE, 'seed-42 run of the final configuration': REF_RUN / 'metrics.csv'}
if DRY:
    EST_CU, NEEDED = 0.0, {}
PREFLIGHT = session.preflight({'notebook': NOTEBOOK, 'profile': PROFILE, 'seeds': SEEDS, 'experiment': EXPERIMENT,
                               'overrides': OVERRIDES}, paths=NEEDED, est_cu=EST_CU, cu_left=CU_LEFT)

In [ ]:
# Cell 3: the shards (dry run: fake world -> fake CP1 run -> pack), copied to the local disk
SHARDS = paths.shards
if DRY:
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', coco_images=40)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True, overrides=cp1_overrides(40))
    out = paths.runs / 'checkpoints' / 'cp1'
    run_cp1(paths, CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20,
                             base_unclear_max=0.2, diffseg_card_splits={'train': 18, 'validation': 12}),
            out_dir=out, cache_dir=paths.cache / 'cp1', progress=None)
    pack.pack(out / 'manifest_cp1.csv.gz', SHARDS, free_bytes=10**12, margin=0,
              shard_bytes=200_000, sample_per_source=5, progress=None)
done = pack.read_done(SHARDS)
assert done, f'no finished shards in {SHARDS}: run notebook 04 first'
print('shards:', done['images'], 'images |', done['mask_rule'])
# each read through Colab's Drive mount costs ~0.25 s: train from a local copy (checked shard by shard)
LOCAL_SHARDS = paths.local_data / 'shards'          # /content/data/shards on Colab
staged = pack.stage_shards(SHARDS, LOCAL_SHARDS, progress=print)
SHARDS = LOCAL_SHARDS
INDEX_SHA256 = pack.file_sha256(SHARDS / 'index.parquet')
print('staged:', staged)

In [ ]:
# Cell 4: A1: CP3's record is reused (the 64-image overfit checks the pipeline, not the seed)
from eviground.train.trainer import NonFiniteLoss, Trainer
if DRY and not A1_FILE.is_file():   # the dry run has no CP3 record: make one with the tiny model
    ot = Trainer(cfg, SHARDS, A1_FILE.parent, seed=42, profile='unified', resume=None)
    A1_FILE.parent.mkdir(parents=True, exist_ok=True)
    A1_FILE.write_text(json.dumps({**ot.overfit(), 'index_sha256': INDEX_SHA256}, indent=1, default=str))
    del ot
OVERFIT = json.loads(A1_FILE.read_text())
assert OVERFIT.get('index_sha256') == INDEX_SHA256, 'the A1 record is for other shards: stop and ask Dhanush'
a1_ok = OVERFIT['acc_pass'] and OVERFIT['mask_pass']
print('A1 (CP3 record, reused):', {k: OVERFIT.get(k) for k in ('acc_reached_at_step', 'mask_f1_reached_at_step', 'acc_pass', 'mask_pass')})
if not a1_ok and not DRY:
    raise RuntimeError(f'the A1 record does not pass: training not started. Send {A1_FILE} to Claude.')

In [ ]:
# Cell 5: training, seed after seed, in the foreground. Re-running the notebook skips finished seeds and resumes the rest.
SUMMARIES, NONFINITE = {}, {}
for seed in SEEDS:
    run = run_for(seed)
    if (run / 'summary.json').is_file():
        SUMMARIES[seed] = json.loads((run / 'summary.json').read_text())
        print(f'seed {seed}: finished earlier, skipped ({run.name})')
        continue
    t = time.time()
    tr = Trainer(cfg, SHARDS, run, seed=seed, profile=PROFILE, resume='auto', progress=True)
    print(f'seed {seed}:', {'steps_per_epoch': tr.steps_per_epoch, 'epochs': tr.epochs,
                            'starting_at_global_step': tr.state['global_step'], 'epoch': tr.state['epoch']})
    try:
        summary = tr.fit()
    except NonFiniteLoss as e:      # S1: last.pt and nonfinite.json are saved; the next seed still runs
        NONFINITE[seed] = str(e)
        summary = tr.summary()
        print(f'seed {seed} STOPPED:', e)
    summary['session_training_minutes'] = round((time.time() - t) / 60, 2)
    if seed not in NONFINITE:
        (run / 'summary.json').write_text(json.dumps(summary, indent=1, default=str))
    SUMMARIES[seed] = summary
    del tr
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
print({s: {k: v.get(k) for k in ('best_epoch', 'best', 'stopped', 'session_training_minutes')} for s, v in SUMMARIES.items()})

In [ ]:
# Cell 6: curves per seed (saved as curves.png in each run folder)
from eviground.eval import cp3
METRICS = 'metrics.csv'  # each run's per-epoch log
def plot_curves(RUN):

    rows = cp3.read_csv(RUN / METRICS)
    steps = cp3.read_csv(RUN / 'steps.csv')
    num = lambda r, k: float(r[k]) if r.get(k) not in (None, '') else float('nan')
    try:
        import matplotlib.pyplot as plt
    except ImportError:          # not installed in the CI dry run; Colab has it
        plt = None
        print('matplotlib not installed: no curves')
    if plt is not None and rows:
        ep = [int(r['epoch']) + 1 for r in rows]
        fig, ax = plt.subplots(2, 2, figsize=(13, 8))
        a = ax[0, 0]
        a.plot([int(s['global_step']) for s in steps], [num(s, 'loss_total') for s in steps],
               lw=0.4, alpha=0.5, label='training loss per step')
        a.plot([int(r['global_step']) for r in rows], [num(r, 'train_total') for r in rows],
               'o-', label='training loss, epoch mean')
        a.set_xlabel('optimizer step'); a.set_title('S1 loss'); a.legend()
        a = ax[0, 1]
        auc = 'val_macro_auc' if PROFILE == 'unified' else 'val_det_auc'
        a.plot(ep, [num(r, 'val_selection') for r in rows], 'o-', label='validation selection')
        a.plot(ep, [num(r, 'train_eval_selection') for r in rows], 's--', label='training subset selection')
        a.plot(ep, [num(r, auc) for r in rows], '^-', label=auc)
        if PROFILE == 'unified':
            for c in ('real', 'generated', 'edited'):
                a.plot(ep, [num(r, f'val_auc_{c}') for r in rows], lw=1, label=f'AUC {c}')
        else:
            a.plot(ep, [num(r, 'val_det_acc') for r in rows], 'v-', label='SD v1.4 accuracy (S3)')
            a.axhline(0.95, ls=':', c='k')
        a.axhline(0.90, ls=':', c='grey'); a.set_xlabel('epoch'); a.set_title('S2 validation'); a.legend(fontsize=7)
        if PROFILE == 'unified':
            a = ax[1, 0]
            for k, lab in (('val_mask_f1', 'validation mask F1 (A4)'), ('train_eval_mask_f1', 'training subset mask F1'),
                           ('val_mask_miou_binary', 'validation binary mIoU (R2)')):
                a.plot(ep, [num(r, k) for r in rows], 'o-', label=lab)
            a.axhline(0.50, ls=':', c='k'); a.axhline(0.2162, ls=':', c='r', label='untrained (CP2)')
            a.set_xlabel('epoch'); a.set_title('A4 / R2 localization'); a.legend(fontsize=7)
            a = ax[1, 1]
            for k in ('val_real_any_pred', 'val_gen_any_pred', 'val_real_pred_area', 'val_gen_pred_area'):
                a.plot(ep, [num(r, k) for r in rows], 'o-', label=k[4:])
            a.axhline(0.20, ls=':', c='k'); a.axhline(0.01, ls=':', c='grey')
            a.set_yscale('symlog', linthresh=0.01); a.set_xlabel('epoch'); a.set_title('A2 / A3 marks on unedited crops')
            a.legend(fontsize=7)
        else:
            ax[1, 0].axis('off'); ax[1, 1].axis('off')
        fig.suptitle(f'{RUN.name}')
        fig.tight_layout()
        fig.savefig(RUN / 'curves.png', dpi=110)
        plt.show()
    print(len(rows), 'epochs logged')

for seed in SEEDS:
    plot_curves(run_for(seed))

In [ ]:
# Cell 7: the CP3 checks per seed, the comparison with seed 42, and the result files
from eviground.eval import fixcompare
REPORTS = {}
ref_rows = cp3.read_csv(REF_RUN / METRICS) if (REF_RUN / METRICS).is_file() else None
for seed in SEEDS:
    run = run_for(seed)
    report = cp3.evaluate_run(run, PROFILE, OVERFIT)
    compare = fixcompare.compare_runs(ref_rows, cp3.read_csv(run / METRICS)) if ref_rows else None
    compute_file = run / 'compute.json'
    report.update({
        'step': f'training health, unified model, final configuration, seed {seed}', 'dry_run': DRY,
        'context_crop_share': CONTEXT_SHARE, 'run_folder': str(run), 'seed': seed, 'branch': BRANCH,
        'compare_with_seed42': compare, 'compare_note': "keys 'cp3' = seed 42 of the final configuration, 'fix' = this seed",
        'a1_note': 'A1 reused from the CP3 record (pipeline check, not seed-specific)',
        'session': info, 'config': CONFIG, 'overrides': OVERRIDES, 'clip_revision': REVISION,
        'index_sha256': INDEX_SHA256, 'summary': SUMMARIES.get(seed), 'nonfinite': NONFINITE.get(seed),
        'compute_run': json.loads(compute_file.read_text()) if compute_file.is_file() else None,
        'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    })
    out = cp3.write_report(report, CK / f'training_health_{PROFILE}_ctx_seed{seed}.json')
    REPORTS[seed] = report
    print(f'seed {seed}: verdict', report['verdict'], '| best epoch', report.get('best_epoch'),
          '| run CU', (report['compute_run'] or {}).get('compute_units'), '| saved', out.name)
    for name, c in report.get('checks', {}).items():
        print(f'   {name:4} {c["status"]:13}', {k: v for k, v in c.items() if k != 'status'})
    for label, v in ((compare or {}).get('metrics') or {}).items():
        print(f"   {label:42} seed 42 {v['cp3']}  seed {seed} {v['fix']}  change {v['change']}")

In [ ]:
# Cell 8: the session's compute record (each run's compute.json covers its GPU training hours), then the
# runtime is released. Run it last: everything above is saved on Drive.
COMPUTE = session.finish(CK, T0, CU_PER_HOUR, name='compute_unified_seeds.json',
                         extra={'git_commit': info['git_commit'], 'gpu': info.get('gpu_name'), 'est_cu': EST_CU,
                                'seeds': SEEDS, 'run_cu': {s: (REPORTS[s]['compute_run'] or {}).get('compute_units') for s in SEEDS}})
RELEASED = session.release(DRY)
if DRY:
    assert RELEASED == 'dry', RELEASED
    assert all((run_for(s) / 'summary.json').is_file() for s in SEEDS)